# nbimplot API Gallery

Public-facing, method-focused reference notebook.

Design goals:
- quick copy/paste snippets
- one primary API focus per example cell
- complete practical coverage of `Plot`, plus `Subplots` and `AlignedPlots`


## Setup


In [1]:
import numpy as np
import nbimplot as ip

rng = np.random.default_rng(2026)

x = np.linspace(0, 20, 2000, dtype=np.float32)
y = (np.sin(1.3 * x) + 0.2 * np.cos(4.1 * x)).astype(np.float32)
y2 = (0.8 * np.sin(1.6 * x + 0.3)).astype(np.float32)

x_short = np.linspace(-4, 4, 400, dtype=np.float32)
y_short = (0.4 * x_short**2 + 0.5 * np.sin(2.5 * x_short)).astype(np.float32)

err = (0.08 + 0.04 * np.abs(np.sin(x_short[:60]))).astype(np.float32)
err_neg = (0.04 + 0.03 * np.abs(np.cos(x_short[:60]))).astype(np.float32)
err_pos = (0.10 + 0.05 * np.abs(np.sin(x_short[:60]))).astype(np.float32)

heat = rng.normal(size=(60, 90)).astype(np.float32)
img_gray = np.linspace(0, 1, 80 * 120, dtype=np.float32).reshape(80, 120)
img_rgb = np.stack([
    np.clip(np.sin(np.linspace(0, 6, 80 * 120)).reshape(80, 120) * 0.5 + 0.5, 0, 1),
    np.clip(np.cos(np.linspace(0, 9, 80 * 120)).reshape(80, 120) * 0.5 + 0.5, 0, 1),
    np.clip(np.sin(np.linspace(0, 12, 80 * 120)).reshape(80, 120) * 0.5 + 0.5, 0, 1),
], axis=-1).astype(np.float32)

pie_vals = np.array([35, 24, 18, 13, 10], dtype=np.float32)
pie_labels = ["A", "B", "C", "D", "E"]


def new_plot(title: str, w: int = 980, h: int = 320) -> ip.Plot:
    return ip.Plot(width=w, height=h, title=title)


## Core Line APIs


In [2]:
p = new_plot("line")
h = p.line("series", y)
p.show()


In [3]:
p = new_plot("line style")
h = p.line("styled", y, color="#1d4ed8", line_weight=1.8, marker="circle", marker_size=4.5)
p.show()


In [4]:
p = new_plot("set_data")
h = p.line("mutable", y)
h.set_data(y2)
p.render()
p.show()


In [5]:
p = new_plot("stream_line")
h = p.stream_line("stream", capacity=3000, initial=np.linspace(0, 1, 100, dtype=np.float32))
p.show()


In [6]:
p = new_plot("append")
h = p.stream_line("stream", capacity=3000, initial=np.linspace(0, 1, 100, dtype=np.float32))
h.append(np.sin(np.linspace(0, 4, 600, dtype=np.float32)).astype(np.float32))
p.render()
p.show()


In [9]:
import time
for i in range(100):
    h.append([i])
    time.sleep(1)
    

KeyboardInterrupt: 

In [10]:
p = new_plot("hide_next_item")
p.line("visible", y)
p.hide_next_item()
p.line("hidden", y2)
p.show()


In [11]:
p = new_plot("render")
p.line("series", y)
p.render()
p.show()


### Realtime Streaming Loop

Timed live-update example for `stream_line` + `append` + `render`.


In [12]:
import time

p = new_plot("realtime stream", 980, 300)
h = p.stream_line("rt", capacity=40_000, initial=np.zeros(1000, dtype=np.float32))
p.show()

phase = 0.0
for _ in range(120):
    c = np.linspace(phase, phase + 0.12, 350, dtype=np.float32)
    v = (np.sin(9.5 * c) + 0.2 * np.cos(2.1 * c) + 0.04 * rng.normal(size=c.size)).astype(np.float32)
    h.append(v)
    p.render()
    phase += 0.12
    time.sleep(0.04)


## Scatter / Step / Stem / Digital


In [13]:
p = new_plot("scatter")
p.scatter("scatter", y_short, x=x_short, size=2.0)
p.show()


In [14]:
p = new_plot("bubbles")
sz = (6 + 20 * np.abs(np.sin(x_short))).astype(np.float32)
p.bubbles("bubbles", y_short, sz, x=x_short)
p.show()


In [15]:
p = new_plot("stairs")
p.stairs("stairs", y_short, x=x_short)
p.show()


In [16]:
p = new_plot("stems")
p.stems("stems", np.abs(y_short), x=x_short)
p.show()


In [17]:
p = new_plot("digital")
d = (np.sin(np.linspace(0, 25, 300)) > 0).astype(np.float32)
p.digital("digital", d)
p.show()


## Bars Family


In [18]:
p = new_plot("bars")
p.bars("bars", np.abs(y_short[:80]), x=np.arange(80, dtype=np.float32), bar_width=0.7)
p.show()


In [19]:
p = new_plot("bars_h")
p.bars_h("bars_h", np.abs(y_short[:80]), y=np.arange(80, dtype=np.float32), bar_height=0.7)
p.show()


In [20]:
p = new_plot("bar_groups")
a = np.abs(np.sin(np.linspace(0, 4, 20))).astype(np.float32)
b = np.abs(np.cos(np.linspace(0, 4, 20))).astype(np.float32)
p.bar_groups(["A", "B"], np.vstack([a, b]), group_size=0.7, shift=0.05)
p.show()


## Area + Uncertainty


In [21]:
p = new_plot("shaded")
lo = (y - 0.2).astype(np.float32)
hi = (y + 0.2).astype(np.float32)
p.shaded("band", lo, hi, x=x, alpha=0.2)
p.line("mid", y)
p.show()


In [22]:
p = new_plot("error_bars symmetric")
p.error_bars("err", y_short[:60], err, x=x_short[:60])
p.show()


In [23]:
p = new_plot("error_bars asymmetric")
p.error_bars("err_asym", y_short[:60], err_neg=err_neg, err_pos=err_pos, x=x_short[:60])
p.show()


In [24]:
p = new_plot("error_bars_h asymmetric")
p.error_bars_h("err_h_asym", x=np.abs(y_short[:60]) + 0.5, y=x_short[:60], err_neg=err_neg, err_pos=err_pos)
p.show()


## Reference Lines


In [25]:
p = new_plot("inf_lines")
p.line("series", y)
p.inf_lines("x refs", np.array([200, 600, 1200], dtype=np.float32), axis="x")
p.show()


In [26]:
p = new_plot("vlines")
p.line("series", y)
p.vlines("vertical", np.array([250, 900, 1500], dtype=np.float32))
p.show()


In [27]:
p = new_plot("hlines")
p.line("series", y)
p.hlines("horizontal", np.array([-0.6, 0.0, 0.6], dtype=np.float32))
p.show()


## Distribution + Grids


In [29]:
p = new_plot("histogram")
p.histogram("hist", rng.normal(size=50_000).astype(np.float32), bins=70)
p.show()


In [30]:
p = new_plot("histogram2d")
xa = rng.normal(size=70_000).astype(np.float32)
ya = (0.5 * xa + rng.normal(scale=0.9, size=xa.size)).astype(np.float32)
p.histogram2d("hist2d", xa, ya, x_bins=72, y_bins=64, label_fmt="", show_colorbar=True, colorbar_label="Count")
p.show()


In [31]:
p = new_plot("heatmap")
p.heatmap("heatmap", heat, label_fmt="", show_colorbar=True, colorbar_label="Z")
p.show()


In [32]:
p = new_plot("image gray")
p.image("gray", img_gray, bounds=((0.0, 0.0), (12.0, 8.0)), tint=(1.0, 1.0, 1.0, 0.95), image_flags=0)
p.show()


In [33]:
p = new_plot("image rgb")
p.image("rgb", img_rgb, bounds=((0.0, 0.0), (12.0, 8.0)), uv0=(0.0, 0.0), uv1=(1.0, 1.0))
p.show()


## Pie / Text / Annotation / Dummy


In [34]:
p = new_plot("pie_chart", 860, 360)
p.pie_chart("pie", pie_vals, labels=pie_labels, x=0.0, y=0.0, radius=1.2, angle0=90.0, label_fmt="%.1f")
p.show()


In [35]:
p = new_plot("text")
p.line("series", y)
p.text("label", float(x[200]), float(y[200]))
p.show()


In [36]:
p = new_plot("annotation")
p.line("series", y)
p.annotation("event", float(x[400]), float(y[400]), offset_x=10, offset_y=-16)
p.show()


In [37]:
p = new_plot("dummy")
p.line("series", y)
p.dummy("legend-item")
p.show()


## Tags + Colormap Widgets


In [38]:
p = new_plot("tag_x")
p.line("series", y)
p.tag_x(6.0, label_fmt="x=%.2f", round_value=False)
p.show()


In [39]:
p = new_plot("tag_y")
p.line("series", y)
p.tag_y(0.0, label_fmt="y=%.2f", round_value=True)
p.show()


In [40]:
p = new_plot("colormap_slider")
p.heatmap("heat", heat, label_fmt="")
p.colormap_slider(label="Colormap T", t=0.45, fmt="%.2f")
p.show()


In [41]:
p = new_plot("colormap_button")
p.heatmap("heat", heat, label_fmt="")
p.colormap_button(label="Cycle", width=76, height=18)
p.show()


In [42]:
p = new_plot("colormap_selector")
p.heatmap("heat", heat, label_fmt="")
p.colormap_selector(label="Choose")
p.show()


## Drag Tools


In [43]:
p = new_plot("drag_line_x")
p.line("series", y)
p.drag_line_x("dx", 4.5, thickness=1.4)
p.show()


In [44]:
p = new_plot("drag_line_y")
p.line("series", y)
p.drag_line_y("dy", 0.2, thickness=1.4)
p.show()


In [45]:
p = new_plot("drag_point")
p.line("series", y)
p.drag_point("dp", 5.0, 0.4, size=5.5)
p.show()


In [46]:
p = new_plot("drag_rect")
p.line("series", y)
p.drag_rect("dr", 4.0, -0.6, 7.0, 0.7)
p.show()


## Drag/Drop Targets


In [47]:
p = new_plot("drag_drop_plot")
p.line("a", y)
p.line("b", y2)
p.drag_drop_plot(source=True, target=True)
p.show()


In [48]:
p = new_plot("drag_drop_axis")
p.line("a", y)
p.drag_drop_axis("x1", source=True, target=True)
p.drag_drop_axis("y1", source=True, target=True)
p.show()


In [49]:
p = new_plot("drag_drop_legend")
p.line("a", y)
p.line("b", y2)
p.drag_drop_legend(target=True)
p.show()


## Axis / View / Plot Controls


In [50]:
p = new_plot("set_axis_scale")
p.set_axis_scale(x="log", y="linear")
xx = np.logspace(0, 3, 1500, dtype=np.float32)
yy = (0.2 * np.log10(xx) + 0.3 * np.sin(np.log(xx))).astype(np.float32)
p.line("log-x", yy)
p.show()


In [51]:
p = new_plot("set_axis_label")
p.line("series", y)
p.set_axis_label("x1", "Samples")
p.set_axis_label("y1", "Amplitude")
p.show()


In [52]:
p = new_plot("set_axis_format")
p.line("series", y)
p.set_axis_format("y1", "%.3f")
p.show()


In [53]:
p = new_plot("set_axis_ticks")
p.line("series", y)
ticks = np.array([0, 400, 800, 1200, 1600, 1999], dtype=np.float32)
p.set_axis_ticks("x1", ticks, labels=[str(int(t)) for t in ticks], keep_default=False)
p.show()


In [54]:
p = new_plot("clear_axis_ticks")
p.line("series", y)
ticks = np.array([0, 500, 1000, 1500, 1999], dtype=np.float32)
p.set_axis_ticks("x1", ticks, labels=[str(int(t)) for t in ticks])
p.clear_axis_ticks("x1")
p.show()


In [55]:
p = new_plot("set_axis_limits_constraints")
p.line("series", y)
p.set_axis_limits_constraints("x1", 0.0, 2000.0)
p.show()


In [56]:
p = new_plot("set_axis_zoom_constraints")
p.line("series", y)
p.set_axis_zoom_constraints("x1", 10.0, 1200.0)
p.show()


In [57]:
p = new_plot("set_axis_link")
p.set_secondary_axes(x2=True)
p.line("x1", y, x_axis="x1", y_axis="y1")
p.line("x2", y2, x_axis="x2", y_axis="y1")
p.set_axis_link("x2", "x1")
p.show()


In [58]:
p = new_plot("set_secondary_axes")
p.set_secondary_axes(x2=True, y2=True)
p.line("primary", y, x_axis="x1", y_axis="y1")
p.line("secondary", y2 * 10 + 30, x_axis="x2", y_axis="y2")
p.show()


In [59]:
p = new_plot("set_time_axis")
base = 1_762_000_000.0
xt = (base + np.arange(2000, dtype=np.float32) * 60.0).astype(np.float32)
yt = (100 + np.cumsum(0.05 * rng.normal(size=xt.size)).astype(np.float32)).astype(np.float32)
p.line("close", yt)
p.set_time_axis("x1")
p.show()


In [60]:
p = new_plot("set_plot_flags")
p.set_plot_flags(no_menus=False, no_box_select=False, crosshairs=True)
p.line("series", y)
p.show()


In [61]:
p = new_plot("set_subplots_config", 1100, 700)
p.set_subplots_config(rows=2, cols=2, flags=0)
p.line("s0", y, subplot_index=0)
p.scatter("s1", y_short, x=x_short, subplot_index=1)
p.histogram("s2", rng.normal(size=20_000).astype(np.float32), subplot_index=2)
p.heatmap("s3", heat, label_fmt="", subplot_index=3)
p.show()


In [62]:
p = new_plot("set_aligned_group", 980, 320)
p.set_aligned_group("api-gallery-group", enabled=True, vertical=True)
p.line("series", y)
p.show()


In [63]:
p = new_plot("set_colormap")
p.set_colormap("Plasma")
p.heatmap("heat", heat, label_fmt="")
p.show()


In [64]:
p = new_plot("set_view")
p.line("series", y)
p.set_view(200.0, 1100.0, -1.5, 1.5)
p.show()


In [65]:
p = new_plot("autoscale")
p.line("series", y)
p.autoscale()
p.show()


## Callback APIs


In [66]:
tool_events = []

def on_tool(_plot, payload):
    tool_events.append(payload)

p = new_plot("on_tool_change")
p.line("series", y)
p.drag_line_x("dx", 5.0)
p.on_tool_change(on_tool)
p.show()
print("Interact with drag tool, then inspect tool_events")


Interact with drag tool, then inspect tool_events


In [67]:
selection_events = []

def on_selection(_plot, payload):
    selection_events.append(payload)

p = new_plot("on_selection_change")
p.line("series", y)
p.on_selection_change(on_selection)
p.show()
print("Box-select on plot, then inspect selection_events")


Box-select on plot, then inspect selection_events


In [68]:
perf_events = []

def on_perf(_plot, payload):
    perf_events.append(payload)

p = new_plot("on_perf_stats")
p.line("series", y)
p.on_perf_stats(on_perf, interval_ms=300)
p.show()
print("Let it run briefly, then inspect perf_events")


Let it run briefly, then inspect perf_events


## Layout APIs: `Subplots` and `AlignedPlots`


In [69]:
sp = ip.Subplots(2, 2, width=1100, height=760, title="Subplots API", link_rows=True, link_cols=True, share_items=True)
sp.subplot(0, 0).line("line", y)
sp.subplot(0, 1).scatter("scatter", y_short, x=x_short)
sp.subplot(1, 0).bars("bars", np.abs(y_short[:80]), x=np.arange(80, dtype=np.float32))
sp.subplot(1, 1).heatmap("heat", heat, label_fmt="")
sp.show()


In [70]:
al = ip.AlignedPlots(2, 1, group_id="aligned-gallery", vertical=True, width=1100, height=700, title="AlignedPlots API")
al.subplot(0, 0).line("top", y)
al.subplot(1, 0).histogram("bottom", rng.normal(size=30_000).astype(np.float32), bins=60)
al.show()


## End

Use this notebook as a compact API lookup.
For richer scenario walkthroughs and benchmark-oriented examples, see:
- `notebooks/nbimplot_examples.ipynb`
- `notebooks/nbimplot_benchmarks.ipynb`
